In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/mda_train_clinical_info.csv")

print(df.shape)
df.head()

(156, 84)


,Unnamed: 0,TCIA code,Sex,Age,Date of Birth,Diag,Site,Histology,Grade,T,...,Pre-RT Imaging Date,Pre-RT Imaging Modality,CT sim date,post-RT imaging date,post-RT imaging modality,Recurrence imaging date,Recurrence imaging modality,Additional imaging date,Additional imaging modality,Additional imaging notes
0,0,HNSCC-01-0001,Female,54,NaN,CA soft palate,Oropharynx,SCC,moderately to poorly diff.,2,...,1998-12-01,PET/CT,1998-12-05,1999-03-27,PET/CT,NaN,NaN,NaN,NaN,NaN
1,2,HNSCC-01-0003,Male,48,NaN,CA tonsil,Oropharynx,SCC,moderately diff.,2,...,2001-10-30,PET/CT,2001-11-20,2002-06-09,PET/CT,NaN,NaN,NaN,NaN,NaN
2,3,HNSCC-01-0004,Male,65,NaN,CA tonsil,Oropharynx,SCC,poorly diff.,4a,...,1996-08-28,PET/CT,1996-08-24,1997-05-11,PET/CT,NaN,NaN,NaN,NaN,NaN
3,4,HNSCC-01-0005,Male,66,NaN,CA tonsil,Oropharynx,SCC,moderately diff.,2,...,NaN,NaN,1998-01-19,2000-03-27,PET/CT,2000-03-27,PET/CT,NaN,NaN,NaN
4,6,HNSCC-01-0007,Male,39,NaN,CA BOT,Oropharynx,SCC,poorly diff.,2,...,1997-05-10,PET/CT,1997-04-29,2000-08-21,PET/CT,NaN,NaN,NaN,NaN,NaN


In [3]:
survival_col = "Survival  (months)"
event_col = "Overall Survival Censor"

print("Survival summary:")
print(df[survival_col].describe())

print("\nCensor values:")
print(df[event_col].value_counts())

Survival summary:
count    156.000000
mean      68.655301
std       27.039964
min        8.933333
25%       52.833333
50%       77.500000
75%       87.158333
max      116.449315
Name: Survival  (months), dtype: float64

Censor values:
Overall Survival Censor
0    113
1     43
Name: count, dtype: int64


In [4]:
from lifelines import CoxPHFitter

model_data = df[
    ["Age", survival_col, event_col]
].dropna().copy()

cox = CoxPHFitter()

cox.fit(
    model_data,
    duration_col=survival_col,
    event_col=event_col
)

cox.print_summary()

<lifelines.CoxPHFitter: fitted with 156 total observations, 113 right-censored observations>
             duration col = 'Survival  (months)'
                event col = 'Overall Survival Censor'
      baseline estimation = breslow
   number of observations = 156
number of events observed = 43
   partial log-likelihood = -199.51
         time fit was run = 2026-10-05 11:17:28 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
Age        0.03      1.04      0.02            0.00            0.07                1.00                1.07

           cmp to    z    p  -log2(p)
covariate                            
Age          0.00 2.05 0.04      4.64
---
Concordance = 0.57
Partial AIC = 401.02
log-likelihood ratio test = 4.19 on 1 df
-log2(p) of ll-ratio test = 4.62

In [5]:
print("Concordance index:")
print(cox.concordance_index_)

Concordance index:
0.5682715074980268


In [6]:
[c for c in df.columns if c in ["Age", "Sex", "Site", "Histology", "Grade", "T", "N", "HPV status"]]

['Sex', 'Age', 'Site', 'Histology', 'Grade', 'T', 'N', 'HPV status']

In [7]:
features = ["Age", "Sex", "Site", "Histology", "Grade", "T", "N", "HPV status"]

model_data = df[
    features + [survival_col, event_col]
].copy()

model_data = pd.get_dummies(
    model_data,
    columns=["Sex", "Site", "Histology", "Grade", "T", "N", "HPV status"],
    drop_first=True
)

model_data = model_data.dropna()

print("Rows used:", len(model_data))
print("Columns used:", model_data.shape[1])

cox_multi = CoxPHFitter(penalizer=0.1)

cox_multi.fit(
    model_data,
    duration_col=survival_col,
    event_col=event_col
)

print("Multivariable C-index:")
print(cox_multi.concordance_index_)

Rows used: 156
Columns used: 23
Multivariable C-index:
0.7598658247829518


In [8]:
simple_features = ["Age", "Sex", "T", "N", "HPV status"]

cv_data = df[
    simple_features + [survival_col, event_col]
].copy()

cv_data = pd.get_dummies(
    cv_data,
    columns=["Sex", "T", "N", "HPV status"],
    drop_first=True
)

cv_data = cv_data.dropna()

print("Rows:", len(cv_data))
print("Columns:", cv_data.shape[1])

Rows: 156
Columns: 18


In [9]:
cv_data = df[
    ["Age", "Sex", "HPV status", survival_col, event_col]
].copy()

cv_data = pd.get_dummies(
    cv_data,
    columns=["Sex", "HPV status"],
    drop_first=True
)

cv_data = cv_data.dropna()

print("Rows:", len(cv_data))
print("Columns:", cv_data.shape[1])
print(cv_data.columns.tolist())

Rows: 156
Columns: 5
['Age', 'Survival  (months)', 'Overall Survival Censor', 'Sex_Male', 'HPV status_positive']


In [10]:
from lifelines.utils import k_fold_cross_validation

cv_model = CoxPHFitter(penalizer=1.0)

scores = k_fold_cross_validation(
    cv_model,
    cv_data,
    duration_col=survival_col,
    event_col=event_col,
    k=5,
    scoring_method="concordance_index"
)

print("5-fold C-index scores:")
print(scores)

print("Mean CV C-index:")
print(sum(scores) / len(scores))

5-fold C-index scores:
[np.float64(0.6889952153110048), np.float64(0.5184049079754601), np.float64(0.6762820512820513), np.float64(0.6973684210526315), np.float64(0.5420560747663551)]
Mean CV C-index:
0.6246213340775005


In [11]:
real_result = pd.DataFrame([{
    "dataset": "MDA clinical",
    "patients": len(cv_data),
    "features": "Age, Sex, HPV status",
    "validation": "5-fold cross-validation",
    "mean_c_index": sum(scores) / len(scores)
}])

real_result.to_csv(
    "../results/mda_real_survival_cv.csv",
    index=False
)

print(real_result)

        dataset  patients              features               validation  \
0  MDA clinical       156  Age, Sex, HPV status  5-fold cross-validation   

   mean_c_index  
0      0.624621  


In [12]:
candidate_features = ["Age", "Sex", "HPV status", "Grade"]

candidate_data = df[
    candidate_features + [survival_col, event_col]
].copy()

candidate_data = pd.get_dummies(
    candidate_data,
    columns=["Sex", "HPV status", "Grade"],
    drop_first=True
)

candidate_data = candidate_data.dropna()

print("Rows:", len(candidate_data))
print("Columns:", candidate_data.shape[1])

Rows: 156
Columns: 10


In [13]:
candidate_model = CoxPHFitter(penalizer=1.0)

candidate_scores = k_fold_cross_validation(
    candidate_model,
    candidate_data,
    duration_col=survival_col,
    event_col=event_col,
    k=5,
    scoring_method="concordance_index"
)

print("Candidate 5-fold C-index scores:")
print(candidate_scores)

print("Candidate mean CV C-index:")
print(sum(candidate_scores) / len(candidate_scores))

Candidate 5-fold C-index scores:
[np.float64(0.7057522123893806), np.float64(0.7650602409638554), np.float64(0.5761421319796954), np.float64(0.4914772727272727), np.float64(0.4807692307692308)]
Candidate mean CV C-index:
0.603840217765887


In [14]:
base_scores = k_fold_cross_validation(
    CoxPHFitter(penalizer=1.0),
    cv_data,
    duration_col=survival_col,
    event_col=event_col,
    k=5,
    scoring_method="concordance_index",
    seed=42
)

grade_scores = k_fold_cross_validation(
    CoxPHFitter(penalizer=1.0),
    candidate_data,
    duration_col=survival_col,
    event_col=event_col,
    k=5,
    scoring_method="concordance_index",
    seed=42
)

print("Base model mean:", sum(base_scores) / len(base_scores))
print("Grade model mean:", sum(grade_scores) / len(grade_scores))

Base model mean: 0.6154980348808646
Grade model mean: 0.5634340456890199


In [15]:
final_result = pd.DataFrame([{
    "dataset": "MDA clinical",
    "patients": len(cv_data),
    "features": "Age, Sex, HPV status",
    "validation": "5-fold cross-validation",
    "seed": 42,
    "mean_c_index": sum(base_scores) / len(base_scores)
}])

final_result.to_csv(
    "../results/mda_real_survival_cv.csv",
    index=False
)

print(final_result)

        dataset  patients              features               validation  \
0  MDA clinical       156  Age, Sex, HPV status  5-fold cross-validation   

   seed  mean_c_index  
0    42      0.615498  
